# 1- Overview of Steps in Data Preparation

Here is a typical pipeline:

1. Data collection

2. Data cleaning

3. Handling missing values

4. Outlier detection & treatment

5. Data transformation / Feature engineering

6. Dimensionality reduction / data reduction

7. Splitting data (train / validation / test)



##  Step 1 — Data Collection

Gather data from various sources: databases, APIs, logs, CSV files, sensors, web scraping, etc.

Ensure relevance: collect features (variables) that are meaningful for your prediction problem.

Consider infrastructure: use ETL pipelines, data warehouses, or data lakes depending on scale and so forth.

Be aware of data silos and fragmentation — you may collect from disconnected systems that need integration.


## Step 2 — Data Cleaning & Quality Checks

Check for errors, duplicates, inconsistencies, malformed values

Validate that data types are consistent (e.g. dates, numeric, strings)

Standardize formats (for example, date format “YYYY-MM-DD”, currency formats)

Remove or correct corrupted records

Check for data imbalance (e.g., class imbalance in classification tasks)


## Step 3 — Handling Missing Values

Missing values are common in real datasets.

Strategies to address missing data:

Deletion / removal: Remove rows or columns with too many missing values (only if safe to do so).

Imputation: Fill missing values with estimates.

For numeric data: mean, median, mode, forward-fill, backward-fill

For categorical data: most frequent value, a “missing” category

Predictive imputation: Use a model to predict missing values based on other features

Flagging: Add indicator features that mark whether a value was missing

Choice depends on context, proportion of missingness, and whether missingness is random or systematic.



## Step 4 — Outliers & Anomalies

Outliers are data points significantly different from others — they can distort model training.

Detection methods: Z-score (standard deviations from mean), IQR (interquartile range), box plots, domain rules.

Handling outliers:

Remove them

Cap (winsorize) — limit to some threshold

Transform (e.g. log transform)

Leave them (if they are valid extreme cases)

Always apply domain knowledge to decide whether an outlier is meaningful or noise.



## Step 5 — Data Transformation & Feature Engineering

Transform data into forms better suited for learning:

Scaling / normalization:

Min-max scaling → scale values to [0, 1]

Standardization (z-score) → mean = 0, std = 1

Encoding categorical variables:

Label encoding

One-hot encoding

Ordinal encoding

Feature construction: deriving new features from existing ones

e.g. from a timestamp, extract day-of-week, hour, month

combine features (ratios, differences, interactions)



## Step 6 — Dimensionality Reduction / Data Reduction

When there are too many features or data is large, reduce complexity without losing critical information.

Techniques:

Feature selection: choose a subset of relevant features

Dimensionality reduction: PCA (Principal Component Analysis), Linear Discriminant Analysis

Aggregation / sampling: combining or summarizing data (e.g. monthly instead of daily)

Record sampling: sub-sample records to reduce size (with care to preserve distribution)

Helps reduce overfitting and computation cost.



## Step 7 — Split Data for Training, Validation, Test

To properly evaluate models, divide data into distinct sets:

Training set: for fitting the model

Validation set: for hyperparameter tuning, model selection

Test set: final evaluation on unseen data

Common splits: 70/30, 80/20











# House Prices Regression Tutorial
Using Kaggle **House Prices – Advanced Regression Techniques** dataset

This notebook covers:
1. Loading data
2. Exploratory data analysis (EDA)
3. Data cleaning & missing value handling
4. Duplicates & outliers
5. Feature engineering & encoding categorical variables
6. Scaling
7. Train/test split
8. Linear Regression model
9. Evaluation


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from scipy import stats

# Load data (assumes 'train.csv' is in Kaggle working directory)
train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')

df = train.copy().dropna(subset=['SalePrice'])


## 1. Initial Exploration & EDA

In [ ]:

print(df.shape)
print(df.info())
print(df.describe())

# Missing values
missing = df.isnull().sum().sort_values(ascending=False)
print(missing[missing > 0].head(20))

# Correlation heatmap
numeric_feats = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_feats.remove('SalePrice')
corr = df[numeric_feats + ['SalePrice']].corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr, cmap='coolwarm', center=0, vmax=0.8)
plt.title('Correlation matrix')
plt.show()


## 2. Data Cleaning & Missing Value Handling

In [ ]:
# Drop high-missing columns
missing_pct = (df.isnull().sum() / len(df)) * 100
high_missing = missing_pct[missing_pct > 80].index.tolist()
df = df.drop(columns=high_missing)

# Separate features AFTER dropping
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols.remove('SalePrice')
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

# Impute
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

df[num_cols] = num_imputer.fit_transform(df[num_cols])
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])


## 3. Handling Duplicates & Outliers

In [ ]:

# Remove duplicates
df = df.drop_duplicates()

# Remove outliers using z-score on GrLivArea
z = np.abs(stats.zscore(df['GrLivArea']))
df = df[z < 3]


### Why didn't we remove outliers then impute ??

## 4. Feature Engineering & Encoding

In [ ]:

# Log transform target
df['SalePrice_log'] = np.log1p(df['SalePrice'])

# Log transform skewed features
skew_feats = df[num_cols].skew().sort_values(ascending=False)
skewed = skew_feats[abs(skew_feats) > 0.75].index.tolist()
for col in skewed:
    df[col] = np.log1p(df[col])

# New feature
df['TotalBath'] = df['FullBath'] + 0.5 * df['HalfBath']

# OneHot encode categorical
encoder = OneHotEncoder(drop='first', sparse=False, handle_unknown='ignore')
encoded = encoder.fit_transform(df[cat_cols])
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(cat_cols))

# Concatenate
df = pd.concat([df.drop(cat_cols, axis=1).reset_index(drop=True), encoded_df], axis=1)


## 5. Scaling & Train/Test Split

In [ ]:

features = df.drop(['SalePrice', 'SalePrice_log'], axis=1)
target = df['SalePrice_log']

scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

X = pd.DataFrame(features_scaled, columns=features.columns)
y = target.copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## 6. Linear Regression Model

In [ ]:

model = LinearRegression()
model.fit(X_train, y_train)


## 7. Evaluation

In [ ]:
y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")



In [ ]:

# Plot predicted vs actual
plt.figure(figsize=(5,5))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.xlabel("Actual log(SalePrice)")
plt.ylabel("Predicted log(SalePrice)")
plt.title("Predicted vs Actual")
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red')
plt.show()



In [ ]:

print(y_train.min(), y_train.max())
print(y_pred.min(), y_pred.max())

In [ ]:
print("NaNs/INFs:", np.isnan(X).sum().sum(), np.isinf(X).sum().sum())
print("Index alignment:", X.index.equals(y.index))


In [ ]:
from sklearn.linear_model import Ridge, Lasso

# Ridge (L2 regularization)
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
y_pred = ridge.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")

## Recap
In this notebook we:
- Cleaned data (missing, duplicates, outliers)
- Engineered features and encoded categoricals
- Scaled features
- Trained Linear Regression
- Evaluated with RMSE and R²
